## Transfer Learning

### Importing Libraries

In [11]:
import torchvision
import torch
from torchvision import transforms
from PIL import Image
import math
import os
import shutil
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn
from torchvision import models
import torch.optim as optim
from torch.optim import Adam


In [2]:
transfer_model=models.resnet50(pretrained=True)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 239MB/s]


In [3]:
for name, param in transfer_model.named_parameters():
  param.requires_grad=False

Sometimes you might need to freeze the BatchNorm layers in the model, as they will be trained to approximate the mean and standard deviation of the dataset that the model was orginally trained on , not the dataset you want to fine tune on.Some of the signal from your data may end up being lost as BatchNorm corrects your input. You can look at model structure and freeze only layers that aren't BatchNorm like this:
```python
for name, param in transfer_model.named_parameters():
  if ("bn" not in name):
    param.requires_grad=False
```


In [4]:
transfer_model.fc=nn.Sequential(nn.Linear(transfer_model.fc.in_features,500),nn.ReLU(),nn.Dropout(),nn.Linear(500,2))

in_features variable allows us to grab the number of activations coming into a layer. You can also  use out_features to discover the activations coming out.These are handy functions for when you are snapping together network like building bricks; if incoming feature on a layer don't maatch the outgoing feature of the previous layer, you will get an runtime error

Learning rate implemenation by Fast AI

In [10]:
def find_lr(model,loss_fn,optimizer,init_value=1e-8,final_value=10):
  number_in_epoch=len(train_loader)-1
  update_step=(final_value/init_value)**(1/number_in_epoch)
  lr=init_value
  optimizer.param_groups[0]['lr']=lr
  best_loss=0.0
  batch_num=0
  losses=[]
  log_lrs=[]
  for data in train_loader:
    batch_num+=1
    inputs,labels=data
    inputs,labels=inputs,labels
    optimizer.zero_grad()
    outputs=model(inputs)
    loss=loss_fn(outputs,labels)
    if batch_num>1 and loss>4*best_loss:
      return log_lrs[10:-5],losses[10:-5]
    if loss<best_loss or batch_num==1:
      best_loss=loss
    losses.append(loss)
    log_lrs.append(math.log10(lr))
    loss.backward()
    optimizer.step()
    lr*=update_step
    optimizer.param_groups[0]['lr']=lr
  return log_lrs[10:-5],losses[10:-5]


### Differential Learning Rates

To get better accuracy during transfer learning we use differential learning rate means we take different learning rate for different layers

In [15]:
## Optimizer for ResNet-50 model
found_lr=0.0001
optimizer=Adam([{"params":transfer_model.layer4.parameters(),"lr":found_lr/3},
                          {"params":transfer_model.layer3.parameters(),"lr":found_lr/9}],lr=found_lr)

In [16]:
unfreeze_layers=[transfer_model.layer3,transfer_model.layer4]
for layer in unfreeze_layers:
  for param in layer.parameters():
    param.requires_grad=True


### Torchvision Transforms

This is used for data augmentation

In [17]:
torchvision.transforms.ColorJitter(brightness=0.5,contrast=0.5,saturation=0.5,hue=0.5)
# ColorJitted randomly changes the brightness, contrast, saturation and hue of the image.
#Brightness-All should be non negativ number between 0 and 1
# hue-float between -0.5 to 0.5


ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=(-0.5, 0.5))

In [19]:
## To flip your image
torchvision.transforms.RandomHorizontalFlip(p=0.5)
torchvision.transforms.RandomVerticalFlip(p=0.5)

RandomVerticalFlip(p=0.5)

In [20]:
# Random Grayscale is a transformation which randomly turns the image to gray scale
torchvision.transforms.RandomGrayscale(p=0.5)

RandomGrayscale(p=0.5)

In [21]:
#RandomCrop and RandomResizeCrop perform cropping on image of size, which can be int of height and width or tuple of different height and widths
torchvision.transforms.RandomCrop(size=(224,224),padding=None,pad_if_needed=False,fill=0,padding_mode="constant")
torchvision.transforms.RandomResizedCrop(size=(224,224),scale=(0.08,1.0),ratio=(3/4,4/3),interpolation=2)
# RandomResizeCrop is using Bilinear interpolation, but you can also select nearest neighbour or bicubic interpolation by chnaging the interpolation parameter

RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)

In [23]:
# If you want to rotate the image between [-degree,degree]
torchvision.transforms.RandomRotation(degrees=(30,70),expand=False,center=None)

RandomRotation(degrees=[30.0, 70.0], interpolation=nearest, expand=False, fill=0)

In [26]:
# Padding Transform
torchvision.transforms.Pad(padding=1,fill=0,padding_mode="constant")

Pad(padding=1, fill=0, padding_mode=constant)

In [28]:
import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode

transform = transforms.RandomAffine(
    degrees=0,
    translate=(0.1, 0.1),
    scale=(0.9, 1.1),
    shear=10,
    interpolation=InterpolationMode.BILINEAR,
    fill=0
)


In [31]:
# To convert Image into HSV
def _random_colour_Space(x):
  output=x.convert("HSV")
  return output
# Another way
colour_transform=transforms.Lambda(lambda x:_random_colour_Space(x))

In [33]:
# chamge image randomly with every epoch
random_colour_transform=torchvision.transforms.RandomApply([colour_transform],p=0.5)

### Custom Transform

In [38]:
'''Code to add random gaussian noise to the tensor
__call__ which is transform pipeline invoked during transformation process
__repr__ which should return a string representation of the transform, can be used for diagnostic purpose.
In the following code, we implement a transform class that adds random Gaussian noise to a tensor. When the class
is initialized, we pass in the mean and standard distribution of the noise we require, and during the __call__ method,
we sample from this distribution and add it to the incoming tensor:'''
class Noise():
  """Adds noise to a tensor.
  >>> transform.Compose([transforms.ToTensor(),Noise(0.1,0.05)),])"""
  def __init__(self,mean,std):
    self.mean=mean
    self.std=std
  def __call__(self,tensor):
    noise=torch.zeros_like(tensor).normal_(mean=self.mean,std=self.std)
    return tensor.add_(noise)
  def __repr__(self):
    repr= f"{self.__class__.__name__}(mean={self.mean}, std={self.std})"
    return repr



In [40]:
transform = transforms.Compose([
    transforms.ToTensor(),
    Noise(0.1, 0.05)
])


In [41]:
transform

Compose(
    ToTensor()
    Noise(mean=0.1, std=0.05)
)

### Ensembling Prediction

In [ ]:
predictions=[m[i].fit(input) for i in models]
avg_prediction=torch.stack(b).mean(0).argmax()